# PDF -> Semantic JSON v2 (Raw + Interpretation)

This notebook is a **v2 copy** focused on semantic transcription instead of PDF replication.

Pipeline:

1. **Stage 1: Raw extraction** (PyMuPDF -> spans, lines, blocks, page geometry)
2. **Stage 2: Document interpretation** (reading order -> semantic content tree)

Target output:

- `schemaVersion: 2.0`
- ordered `document.content[]` with stable `id`s (`item_001`, `item_002`, ...)
- semantic relationships (`previous`, `next`, `role`)
- spacing/indentation categories for renderer/editor behavior
- confidence + `review_required` for uncertain items

In [4]:
%pip install -q pymupdf

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [5]:
import html
import json
import re
import statistics
from dataclasses import dataclass
from pathlib import Path

import fitz  # PyMuPDF

print("PyMuPDF ready")

PyMuPDF ready


## Configuration

Set the PDF file and output folder.

In [6]:
# Update this if your PDF filename is different.
pdf_path = Path("Statistics.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = output_dir / f"{pdf_path.stem}.raw.v2.json"
semantic_output_path = output_dir / f"{pdf_path.stem}.semantic.v2.json"
html_output_path = output_dir / f"{pdf_path.stem}.semantic.v2.html"

print("PDF:", pdf_path.resolve())
print("Raw output:", raw_output_path.resolve())
print("Semantic output:", semantic_output_path.resolve())
print("HTML preview:", html_output_path.resolve())

PDF: /Users/jonathansanz/LUNA_2/auxiliary/PDF_JS_Jupyter/Statistics.pdf
Raw output: /Users/jonathansanz/LUNA_2/auxiliary/PDF_JS_Jupyter/output/Statistics.raw.v2.json
Semantic output: /Users/jonathansanz/LUNA_2/auxiliary/PDF_JS_Jupyter/output/Statistics.semantic.v2.json
HTML preview: /Users/jonathansanz/LUNA_2/auxiliary/PDF_JS_Jupyter/output/Statistics.semantic.v2.html


## Stage 1: Raw extraction

This stage is coordinate-aware and information-preserving.

Output includes:

- spans
- lines
- blocks
- page metadata

In [12]:
doc = fitz.open(pdf_path)

In [14]:
doc[0].get_text("dict")  # Get the text of the first page as a dictionary

{'width': 595.2000122070312,
 'height': 841.9199829101562,
 'blocks': [{'number': 0,
   'type': 0,
   'bbox': (72.0, 35.291969299316406, 169.6156768798828, 49.943965911865234),
   'lines': [{'spans': [{'size': 12.0,
       'flags': 6,
       'bidi': 0,
       'char_flags': 16,
       'font': 'Aptos-Italic',
       'color': 0,
       'alpha': 255,
       'ascender': 0.9390000104904175,
       'descender': -0.28200000524520874,
       'text': 'Pre-MBA Statistics ',
       'origin': (72.0, 46.559967041015625),
       'bbox': (72.0,
        35.291969299316406,
        169.6156768798828,
        49.943965911865234)}],
     'wmode': 0,
     'dir': (1.0, 0.0),
     'bbox': (72.0,
      35.291969299316406,
      169.6156768798828,
      49.943965911865234)}]},
  {'number': 1,
   'type': 0,
   'bbox': (72.0, 791.77197265625, 303.3261413574219, 806.6639404296875),
   'lines': [{'spans': [{'size': 12.0,
       'flags': 4,
       'bidi': 0,
       'char_flags': 16,
       'font': 'Aptos',
       '

In [7]:
def r2(value):
    return round(float(value), 2)


def rect_to_list(rect_like):
    return [r2(rect_like[0]), r2(rect_like[1]), r2(rect_like[2]), r2(rect_like[3])]


def extract_raw_elements(pdf_path: Path) -> dict:
    doc = fitz.open(pdf_path)

    pages = []
    all_spans = []
    all_lines = []
    all_blocks = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_number = page_index + 1
        page_dict = page.get_text("dict")

        page_info = {
            "page": page_number,
            "width": r2(page.rect.width),
            "height": r2(page.rect.height),
            "rotation": int(page.rotation),
        }
        pages.append(page_info)

        for block_idx, block in enumerate(page_dict.get("blocks", []), start=1):
            block_type = block.get("type", -1)
            block_id = f"p{page_number:03d}_b{block_idx:03d}"

            if block_type != 0:
                # Keep non-text blocks as raw placeholders for downstream logic.
                all_blocks.append({
                    "id": block_id,
                    "page": page_number,
                    "type": "non_text",
                    "bbox": rect_to_list(block.get("bbox", [0, 0, 0, 0])),
                    "raw_type": block_type,
                })
                continue

            line_records = []
            block_text_parts = []

            for line_idx, line in enumerate(block.get("lines", []), start=1):
                line_id = f"{block_id}_l{line_idx:03d}"
                line_bbox = rect_to_list(line.get("bbox", [0, 0, 0, 0]))

                span_ids = []
                span_text_parts = []

                for span_idx, span in enumerate(line.get("spans", []), start=1):
                    span_id = f"{line_id}_s{span_idx:03d}"
                    span_text = span.get("text", "")
                    span_text_parts.append(span_text)
                    span_ids.append(span_id)

                    all_spans.append({
                        "id": span_id,
                        "page": page_number,
                        "block_id": block_id,
                        "line_id": line_id,
                        "text": span_text,
                        "bbox": rect_to_list(span.get("bbox", [0, 0, 0, 0])),
                        "font": span.get("font", ""),
                        "size": r2(span.get("size", 0)),
                        "flags": int(span.get("flags", 0)),
                        "color": int(span.get("color", 0)),
                    })

                line_text = "".join(span_text_parts).strip()
                if line_text:
                    block_text_parts.append(line_text)

                line_record = {
                    "id": line_id,
                    "page": page_number,
                    "block_id": block_id,
                    "text": line_text,
                    "bbox": line_bbox,
                    "span_ids": span_ids,
                }
                line_records.append(line_record)
                all_lines.append(line_record)

            block_text = "\n".join([t for t in block_text_parts if t]).strip()
            all_blocks.append({
                "id": block_id,
                "page": page_number,
                "type": "text",
                "bbox": rect_to_list(block.get("bbox", [0, 0, 0, 0])),
                "text": block_text,
                "line_ids": [line["id"] for line in line_records],
            })

    doc.close()

    return {
        "schemaVersion": "2.0-raw",
        "source": {
            "filename": pdf_path.name
        },
        "pages": pages,
        "spans": all_spans,
        "lines": all_lines,
        "blocks": all_blocks,
    }

## Stage 2: Reading order + semantic document tree

This stage transforms raw items into semantic content:

- heading
- paragraph
- equation
- list / list_item
- image / page_break

Coordinates are converted into inferred structure: ordering, spacing, indentation, and relationships.

In [8]:
EQUATION_RE = re.compile(r"[=+\-*/^]|\\frac|\\sum|\\int|\\bar|√|∑")

BULLET_RE = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s+")





def normalize_ws(text: str) -> str:

    return re.sub(r"\s+", " ", text or "").strip()





def merge_lines_to_paragraph(lines: list[str]) -> str:

    merged = []

    for line in lines:

        line = line.strip()

        if not line:

            continue

        if merged and merged[-1].endswith("-"):

            merged[-1] = merged[-1][:-1] + line

        else:

            merged.append(line)

    return normalize_ws(" ".join(merged))





def classify_spacing(y_gap: float, median_height: float) -> str:

    if median_height <= 0:

        return "unknown"

    ratio = y_gap / median_height

    if ratio < 0.35:

        return "none"

    if ratio < 1.15:

        return "normal"

    if ratio < 1.8:

        return "medium"

    return "large"





def infer_indentation(x0: float, page_left: float, median_indent_step: float = 18.0) -> dict:

    delta = max(0.0, x0 - page_left)

    level = int(round(delta / median_indent_step))

    indent_type = "none" if level == 0 else "left"

    return {

        "level": level,

        "type": indent_type,

    }





def looks_like_equation(text: str) -> bool:

    t = normalize_ws(text)

    if not t:

        return False

    if EQUATION_RE.search(t) is None:

        return False

    words = t.split()

    symbol_count = len(re.findall(r"[=+\-*/^()]", t))

    return symbol_count >= 2 and len(words) <= 24





def split_inline_equations(text: str):

    # Heuristic: turn parenthesized math-like expressions into inline_equation nodes.

    pattern = re.compile(r"\(([^()]*[=+\-*/^][^()]*)\)")

    parts = []

    pos = 0

    for match in pattern.finditer(text):

        start, end = match.span()

        if start > pos:

            parts.append({"type": "text", "text": text[pos:start]})

        parts.append({"type": "inline_equation", "latex": normalize_ws(match.group(1))})

        pos = end



    if pos < len(text):

        parts.append({"type": "text", "text": text[pos:]})



    compact = [p for p in parts if p.get("text", "").strip() or p.get("type") == "inline_equation"]

    return compact if any(p["type"] == "inline_equation" for p in compact) else None





def next_item_id(counter: int) -> str:

    return f"item_{counter:03d}"





def build_semantic_document(raw: dict, pdf_path: Path) -> dict:

    lines_by_id = {line["id"]: line for line in raw["lines"]}

    spans_by_block = {}

    for span in raw["spans"]:

        spans_by_block.setdefault(span["block_id"], []).append(span)



    text_blocks = [b for b in raw["blocks"] if b.get("type") == "text"]



    # Reading order: page asc, top asc, left asc.

    text_blocks.sort(key=lambda b: (b["page"], b["bbox"][1], b["bbox"][0]))



    line_heights = []

    for line in raw["lines"]:

        y0, y1 = line["bbox"][1], line["bbox"][3]

        line_heights.append(max(0.1, y1 - y0))

    median_height = statistics.median(line_heights) if line_heights else 12.0



    spans_sizes = [s["size"] for s in raw["spans"] if s.get("size", 0) > 0]

    median_font = statistics.median(spans_sizes) if spans_sizes else 12.0



    page_left_by_page = {p["page"]: 0.0 for p in raw["pages"]}



    content = []

    id_counter = 1



    previous_item = None

    previous_block = None



    for block in text_blocks:

        block_lines = [lines_by_id[lid] for lid in block.get("line_ids", []) if lid in lines_by_id]

        block_lines = sorted(block_lines, key=lambda line: (line["bbox"][1], line["bbox"][0]))

        line_texts = [line["text"] for line in block_lines if line["text"].strip()]



        if not line_texts:

            continue



        page_no = block["page"]



        if previous_block is not None and previous_block["page"] != page_no:

            page_break = {

                "id": next_item_id(id_counter),

                "type": "page_break",

                "position": len(content) + 1,

                "page": page_no,

                "relationship": {

                    "previous": previous_item["id"] if previous_item else None,

                    "next": None,

                    "role": "page_transition",

                },

            }

            content.append(page_break)

            if previous_item is not None:

                previous_item["relationship"]["next"] = page_break["id"]

            previous_item = page_break

            id_counter += 1



        paragraph_text = merge_lines_to_paragraph(line_texts)

        if not paragraph_text:

            continue



        first_line = block_lines[0]

        x0 = first_line["bbox"][0]

        y0 = first_line["bbox"][1]



        candidate_spans = spans_by_block.get(block["id"], [])

        block_font_size = statistics.median([s["size"] for s in candidate_spans]) if candidate_spans else median_font



        is_heading = (

            len(paragraph_text.split()) <= 14

            and block_font_size >= median_font * 1.12

            and paragraph_text[-1] not in {":", ";", ","}

        )



        is_list_item = BULLET_RE.match(paragraph_text) is not None

        is_equation = looks_like_equation(paragraph_text) and not is_list_item



        item_type = "paragraph"

        role = "body"

        confidence = 0.88



        if is_heading:

            item_type = "heading"

            role = "section_title"

            confidence = 0.84

        elif is_equation:

            item_type = "equation"

            role = "explains_previous_text"

            confidence = 0.68

        elif is_list_item:

            item_type = "list_item"

            role = "enumeration"

            confidence = 0.9



        spacing = "normal"

        if previous_block is not None and previous_block["page"] == block["page"]:

            y_gap = y0 - previous_block["bbox"][3]

            spacing = classify_spacing(y_gap, median_height)



        indentation = infer_indentation(x0, page_left_by_page.get(page_no, 0.0))



        item = {

            "id": next_item_id(id_counter),

            "type": item_type,

            "position": len(content) + 1,

            "page": page_no,

            "relationship": {

                "previous": previous_item["id"] if previous_item else None,

                "next": None,

                "role": role,

            },

            "spacing": {

                "previous_to_current": spacing

            },

            "indentation": indentation,

            "confidence": round(confidence, 2),

            "review_required": confidence < 0.75,

            "provenance": {

                "block_id": block["id"],

                "line_ids": block.get("line_ids", []),

            },

        }



        if item_type == "heading":

            item["level"] = 1

            item["text"] = paragraph_text

        elif item_type == "equation":

            item["display"] = True

            item["latex"] = paragraph_text

        else:

            inline_content = split_inline_equations(paragraph_text)

            if inline_content:

                item["content"] = inline_content

                item["text"] = paragraph_text

            else:

                item["text"] = paragraph_text



        if previous_item is not None:

            previous_item["relationship"]["next"] = item["id"]



        content.append(item)

        previous_item = item

        previous_block = block

        id_counter += 1



    title = None

    for item in content:

        if item.get("type") == "heading":

            title = item.get("text")

            break



    return {

        "schemaVersion": "2.0",

        "document": {

            "metadata": {

                "title": title or pdf_path.stem,

                "source_file": pdf_path.name,

                "page_count": len(raw.get("pages", [])),

            },

            "content": content,

        },

    }


In [9]:
raw_v2 = extract_raw_elements(pdf_path)
semantic_v2 = build_semantic_document(raw_v2, pdf_path)

raw_output_path.write_text(json.dumps(raw_v2, ensure_ascii=False, indent=2), encoding="utf-8")
semantic_output_path.write_text(json.dumps(semantic_v2, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved raw extraction:", raw_output_path)
print("Saved semantic tree:", semantic_output_path)
print("Semantic content items:", len(semantic_v2["document"]["content"]))

Saved raw extraction: output/Statistics.raw.v2.json
Saved semantic tree: output/Statistics.semantic.v2.json
Semantic content items: 66


In [10]:
def semantic_preview(doc: dict, limit: int = 20):
    items = doc["document"]["content"][:limit]
    for item in items:
        payload = item.get("text") or item.get("latex") or item.get("type")
        print(f"{item['position']:>3}. {item['id']} [{item['type']}] :: {str(payload)[:120]}")


semantic_preview(semantic_v2, limit=30)

risks = [i for i in semantic_v2["document"]["content"] if i.get("review_required")]
print("\nReview-required items:", len(risks))
for item in risks[:10]:
    snippet = item.get("text") or item.get("latex") or ""
    print(f" - {item['id']} ({item['type']}, conf={item['confidence']}): {snippet[:100]}")

  1. item_001 [paragraph] :: Pre-MBA Statistics
  2. item_002 [heading] :: 1. Central tendency - mean, median
  3. item_003 [paragraph] :: Median is less sensitive to outliers than the mean ( 𝑥̅). Median nicer to report when extreme outliers. If median and me
  4. item_004 [equation] :: The pth quantile is the number such that p% of observations are below, (1-p)% above
  5. item_005 [heading] :: 2. Sample spread - variance, standard deviation
  6. item_006 [paragraph] :: The average of the di;erence from each variable observation to the mean is always 0. The size of individual di;erences b
  7. item_007 [paragraph] :: " measures spread, is the average squared distance of the mean:
  8. item_008 [paragraph] :: Sample variance 𝑆!
  9. item_009 [paragraph] :: $
 10. item_010 [equation] :: 1 " = 𝑛−1 )(𝑥# −𝑥̅)"
 11. item_011 [paragraph] :: 𝑆!
 12. item_012 [paragraph] :: #%&
 13. item_013 [list_item] :: • Note we square 𝑥# −𝑥̅ to ensure the number will be positive. The larger the value,
 14

In [11]:
def render_semantic_html(doc: dict) -> str:
    items = doc["document"]["content"]

    css = """
    <style>
      body { font-family: Georgia, serif; margin: 2rem auto; max-width: 900px; line-height: 1.5; }
      .item { margin: 0.35rem 0; padding: 0.35rem 0.45rem; border-radius: 6px; }
      .item:hover { background: #f8f9fb; }
      .heading { font-weight: 700; margin-top: 1.2rem; }
      .equation { font-family: 'Times New Roman', serif; background: #f2f4f8; }
      .review { outline: 2px solid #ffbf47; }
      .meta { font-family: ui-monospace, Menlo, monospace; color: #6a7280; font-size: 12px; }
      .inline-equation { font-style: italic; background: #eef2ff; padding: 0 0.2rem; border-radius: 3px; }
      .page-break { border-top: 2px dashed #d0d7de; margin: 1.2rem 0; }
    </style>
    """

    html_parts = ["<html><head><meta charset='utf-8'>", css, "</head><body>"]
    html_parts.append(f"<h1>{html.escape(doc['document']['metadata'].get('title', 'Untitled'))}</h1>")

    for item in items:
        classes = ["item", item["type"]]
        if item.get("review_required"):
            classes.append("review")

        data_attr = f"data-item-id='{html.escape(item['id'])}'"
        meta = f"<div class='meta'>{item['id']} | {item['type']} | conf={item.get('confidence', 1.0)}</div>"

        if item["type"] == "page_break":
            html_parts.append("<div class='page-break'></div>")
            continue

        if item["type"] == "heading":
            html_parts.append(
                f"<div class='{' '.join(classes)}' {data_attr} contenteditable='true'>"
                f"<h2>{html.escape(item.get('text', ''))}</h2>{meta}</div>"
            )
        elif item["type"] == "equation":
            html_parts.append(
                f"<div class='{' '.join(classes)}' {data_attr} contenteditable='true'>"
                f"<div>{html.escape(item.get('latex', ''))}</div>{meta}</div>"
            )
        else:
            if "content" in item:
                fragments = []
                for frag in item["content"]:
                    if frag.get("type") == "inline_equation":
                        fragments.append(f"<span class='inline-equation'>{html.escape(frag.get('latex', ''))}</span>")
                    else:
                        fragments.append(html.escape(frag.get("text", "")))
                text_html = "".join(fragments)
            else:
                text_html = html.escape(item.get("text", ""))

            html_parts.append(
                f"<div class='{' '.join(classes)}' {data_attr} contenteditable='true'>"
                f"<p>{text_html}</p>{meta}</div>"
            )

    html_parts.append("</body></html>")
    return "".join(html_parts)


semantic_html = render_semantic_html(semantic_v2)
html_output_path.write_text(semantic_html, encoding="utf-8")

print("Saved editable HTML preview:", html_output_path)

Saved editable HTML preview: output/Statistics.semantic.v2.html


## Notes

- Stage 1 is intentionally low-level and coordinate-preserving.
- Stage 2 is intentionally semantic and editor-oriented.
- The equation and heading detectors are heuristic and should be tuned with review data.
- Every semantic item has a stable `id` so the review UI can target specific nodes.